# 2. 청킹
Blob Storage에서 텍스트를 불러와 두 가지 전략으로 청킹 후 비교

In [ ]:
import os
import json
from dotenv import load_dotenv
from azure.storage.blob import BlobServiceClient

load_dotenv()

STORAGE_CONNECTION_STRING = os.getenv('AZURE_STORAGE_CONNECTION_STRING')
CONTAINER_NAME = os.getenv('AZURE_STORAGE_CONTAINER_NAME')

TARGET_YEAR = 2023
TARGET_COMPANIES = ['삼성전자', 'SK하이닉스', '현대자동차', 'NAVER', '카카오']

print('환경변수 로드 완료')

## Blob에서 텍스트 불러오기

In [ ]:
def load_from_blob(company_name: str) -> str | None:
    blob_service = BlobServiceClient.from_connection_string(STORAGE_CONNECTION_STRING)
    blob_name = f'{company_name}_{TARGET_YEAR}.txt'
    blob_client = blob_service.get_blob_client(container=CONTAINER_NAME, blob=blob_name)

    try:
        data = blob_client.download_blob().readall()
        return data.decode('utf-8')
    except Exception as e:
        print(f'  {company_name} 로드 실패: {e}')
        return None

# 테스트
sample_text = load_from_blob('삼성전자')
print(f'삼성전자 텍스트 길이: {len(sample_text):,}자')

## 전략 1: Fixed-size 청킹 (베이스라인)

In [ ]:
def fixed_size_chunk(text: str, chunk_size: int = 500, overlap: int = 50) -> list[dict]:
    """
    chunk_size: 청크당 글자 수
    overlap: 앞 청크와 겹치는 글자 수 (문맥 유지)
    """
    chunks = []
    start = 0
    idx = 0

    while start < len(text):
        end = start + chunk_size
        chunk_text = text[start:end]
        chunks.append({
            'chunk_id': idx,
            'strategy': 'fixed',
            'text': chunk_text,
            'char_count': len(chunk_text),
        })
        start += chunk_size - overlap
        idx += 1

    return chunks

fixed_chunks = fixed_size_chunk(sample_text)
print(f'Fixed-size 청크 수: {len(fixed_chunks)}')
print(f'\n--- 샘플 청크 ---')
print(fixed_chunks[0]['text'])

## 전략 2: 섹션 헤더 기반 청킹

In [ ]:
import re

# 사업보고서 주요 섹션 키워드
SECTION_KEYWORDS = [
    '회사의 개요',
    '사업의 내용',
    # '재무에 관한 사항',
    '위험관리',
    '임원 및 직원',
    '주주에 관한 사항',
    '이해관계자와의 거래',
    '그 밖에 투자자 보호',
]

def section_based_chunk(text: str, max_chunk_size: int = 2000) -> list[dict]:
    """
    섹션 키워드로 분할 후 max_chunk_size 초과 시 추가 분할
    """
    # 섹션 키워드 위치 탐색
    pattern = '|'.join(re.escape(k) for k in SECTION_KEYWORDS)
    matches = list(re.finditer(pattern, text))

    if not matches:
        # 섹션 못 찾으면 fixed로 fallback
        print('  섹션 키워드 미발견 → fixed-size fallback')
        return fixed_size_chunk(text)

    # 섹션 경계로 분할
    boundaries = [m.start() for m in matches] + [len(text)]
    raw_sections = []
    for i in range(len(boundaries) - 1):
        section_text = text[boundaries[i]:boundaries[i+1]].strip()
        section_name = matches[i].group()
        raw_sections.append((section_name, section_text))

    # 섹션이 너무 길면 추가 분할
    chunks = []
    idx = 0
    for section_name, section_text in raw_sections:
        if len(section_text) <= max_chunk_size:
            chunks.append({
                'chunk_id': idx,
                'strategy': 'section',
                'section': section_name,
                'text': section_text,
                'char_count': len(section_text),
            })
            idx += 1
        else:
            # 긴 섹션은 fixed-size로 추가 분할
            sub_chunks = fixed_size_chunk(section_text, chunk_size=max_chunk_size, overlap=100)
            for sc in sub_chunks:
                chunks.append({
                    'chunk_id': idx,
                    'strategy': 'section',
                    'section': section_name,
                    'text': sc['text'],
                    'char_count': sc['char_count'],
                })
                idx += 1

    return chunks

section_chunks = section_based_chunk(sample_text)
print(f'Section-based 청크 수: {len(section_chunks)}')
print(f'\n--- 섹션 목록 ---')
seen = set()
for c in section_chunks:
    s = c.get('section', 'N/A')
    if s not in seen:
        print(f'  - {s}')
        seen.add(s)

## 청킹 전략 비교

In [ ]:
print('=== 청킹 전략 비교 (삼성전자) ===')
print(f'원본 텍스트 길이: {len(sample_text):,}자')
print()
print(f'[Fixed-size]')
print(f'  청크 수: {len(fixed_chunks)}')
print(f'  평균 청크 크기: {sum(c["char_count"] for c in fixed_chunks) // len(fixed_chunks):,}자')
print()
print(f'[Section-based]')
print(f'  청크 수: {len(section_chunks)}')
print(f'  평균 청크 크기: {sum(c["char_count"] for c in section_chunks) // len(section_chunks):,}자')
print(f'  발견된 섹션: {len(seen)}개')

## 전체 기업 청킹 후 JSON 저장 (Blob)

In [ ]:
def upload_chunks_to_blob(company_name: str, chunks: list[dict], strategy: str):
    blob_service = BlobServiceClient.from_connection_string(STORAGE_CONNECTION_STRING)
    blob_name = f'chunks/{company_name}_{TARGET_YEAR}_{strategy}.json'
    blob_client = blob_service.get_blob_client(container=CONTAINER_NAME, blob=blob_name)

    # 회사명 메타데이터 추가
    for c in chunks:
        c['company'] = company_name
        c['year'] = TARGET_YEAR

    blob_client.upload_blob(
        json.dumps(chunks, ensure_ascii=False, indent=2).encode('utf-8'),
        overwrite=True
    )
    print(f'  저장 완료: {blob_name} ({len(chunks)}개 청크)')


for company in TARGET_COMPANIES:
    print(f'[{company}] 청킹 중...')
    text = load_from_blob(company)
    if not text:
        continue

    # 두 전략 모두 저장
    fixed = fixed_size_chunk(text)
    upload_chunks_to_blob(company, fixed, 'fixed')

    section = section_based_chunk(text)
    upload_chunks_to_blob(company, section, 'section')

    print()

print('=== 청킹 완료 ===')